In [1]:
import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

True

Chunk Index 
- Full-text
- vector

Entity Index
- Full-text
- vector

In [2]:
from langchain_community.graphs import Neo4jGraph

print("[System] Neo4j 데이터베이스 연결 시도 중...")

try:
    # Neo4j 그래프 객체 초기화 및 연결
    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI"),
        username=os.environ.get("NEO4J_USERNAME"),
        password=os.environ.get("NEO4J_PASSWORD")
    )
    
    # 스키마 새로고침을 통해 실제 DB 통신 확인
    graph.refresh_schema()
    
    print("[System] Neo4j 연결 성공 및 스키마 로드 완료.")
    print("\n[현재 DB 스키마 정보]")
    print("-" * 30)
    print(graph.schema)
    print("-" * 30)

    # Index Schema
    index_queries = [
        """
        CREATE FULLTEXT INDEX chunk_text_idx IF NOT EXISTS 
        FOR (c:Chunk) ON EACH [c.text]
        """,
        """
        CREATE VECTOR INDEX chunk_vector_idx IF NOT EXISTS 
        FOR (c:Chunk) ON (c.embedding)
        OPTIONS {indexConfig: {
        `vector.dimensions`: 1536,
        `vector.similarity_function`: 'cosine'
        }}
        """,
        """
        CREATE VECTOR INDEX entity_vector_idx IF NOT EXISTS 
        FOR (c:Entity) ON (c.embedding)
        OPTIONS {indexConfig: {
        `vector.dimensions`: 1536,
        `vector.similarity_function`: 'cosine'
        }}
        """
    ]

    print("[System] Neo4j 인덱스 생성을 시작합니다...")

    # 리스트에 담긴 쿼리를 하나씩 순회하며 개별 실행
    for i, query in enumerate(index_queries, 1):
        try:
            graph.query(query)
            print(f"[System] 인덱스 {i}/3 생성(또는 확인) 완료.")
        except Exception as e:
            print(f"[Error] 인덱스 {i} 실행 중 오류 발생: {e}")

    print("[System] 모든 인덱스 설정 작업이 완료되었습니다.")

except Exception as e:
    print(f"[Error] Neo4j 연결 실패: {e}")
    print("[System] Docker 컨테이너가 정상적으로 실행 중인지 확인해 주세요.")

[System] Neo4j 데이터베이스 연결 시도 중...


C:\Users\dldyd\AppData\Local\Temp\ipykernel_17552\830354817.py:7: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


[System] Neo4j 연결 성공 및 스키마 로드 완료.

[현재 DB 스키마 정보]
------------------------------
Node properties:

Relationship properties:

The relationships:

------------------------------
[System] Neo4j 인덱스 생성을 시작합니다...
[System] 인덱스 1/3 생성(또는 확인) 완료.
[System] 인덱스 2/3 생성(또는 확인) 완료.
[System] 인덱스 3/3 생성(또는 확인) 완료.
[System] 모든 인덱스 설정 작업이 완료되었습니다.


In [3]:
# 데이터베이스에 실제로 생성된 인덱스 목록을 조회하는 쿼리입니다.
check_index_query = "SHOW INDEXES"

try:
    result = graph.query(check_index_query)
    
    print("\n[현재 Neo4j에 등록된 인덱스 목록]")
    print("-" * 50)
    for record in result:
        name = record.get('name')
        type_ = record.get('type')
        state = record.get('state')
        labels = record.get('labelsOrTypes')
        properties = record.get('properties')
        
        # 시스템 기본 인덱스 제외하고 출력
        if not name.startswith("index_"):
            print(f"이름: {name}")
            print(f"  - 타입: {type_}")
            print(f"  - 상태: {state}")
            print(f"  - 대상: {labels} {properties}\n")
    print("-" * 50)
except Exception as e:
    print(f"[Error] 인덱스 조회 중 오류 발생: {e}")


[현재 Neo4j에 등록된 인덱스 목록]
--------------------------------------------------
이름: chunk_text_idx
  - 타입: FULLTEXT
  - 상태: ONLINE
  - 대상: ['Chunk'] ['text']

이름: chunk_vector_idx
  - 타입: VECTOR
  - 상태: ONLINE
  - 대상: ['Chunk'] ['embedding']

이름: entity_vector_idx
  - 타입: VECTOR
  - 상태: ONLINE
  - 대상: ['Entity'] ['embedding']

--------------------------------------------------
